(User_MolecularSystem_Selection)=
# Selecting elements

Selection is one of the most important features in tools that work with molecular systems.

A **selection** is a query that picks a subset of a molecular system. MolSysViewer relies on MolSysMT to parse and evaluate selection strings.

If you are not familiar with MolSysMT selections yet, start with the MolSysMT tutorial: [Selecting elements](https://www.uibcdf.org/molsysmt/content/user/tools/basic/select). It is the main MolSysMT concept you need to use MolSysViewer smoothly, and it contains many more examples than this page.

In MolSysViewer you will encounter selections in two places:

- **Viewer operations** (hide/show/isolate/zoom, creating regions, etc.).
- **Query helpers** (`view.whole.select`, `view.whole.info`, `view.whole.get`, and their `region` counterparts).

In all cases, selection strings are evaluated through `view.whole.select(...)`, so this page focuses on how to write selections and how to use `view.whole.select(...)` effectively.


In [1]:
import molsysviewer as msv

view = msv.demo["181L"]
view.show()

In [2]:
from molsysviewer.thirds.jupyter import load_html_in_notebook

load_html_in_notebook("../../../_static/views/demo_181L.html")

`view.whole.select(...)` returns indices of the selected elements. By default, it returns **atom indices**, but you can select at other levels by changing `element` to `group`, `molecule`, `chain`, `entity`, and so on.

MolSysMT supports multiple selection syntaxes. In MolSysViewer you control this with the `syntax` argument (default: `"MolSysMT"`).

If you prefer another syntax, check which ones MolSysMT supports in [Selection syntaxes](https://www.uibcdf.org/molsysmt/content/user/intro/selection_syntaxes).

The MolSysMT selection language is rich; what follows is a brief, practical summary. For deeper details, examples, and corner cases, see the MolSysMT page linked above.


## MolSysMT selection syntax

MolSysMT has its own selection syntax based on the names of the attributes of the elements (atoms, groups, molecules, etc.). Its basic rules are briefly summarized in this section.

These are the words you can use to refer to element attributes in MolSysMT selection strings:

```{include} ../_snippets/attributes_table.md
```


The plural forms of the former words are also accepted. For example:


| Word | Equivalent |
|---|---|
| "atom_indices" | "atom_index" |
| "component_names" | "component_name" |
| "occupancies" | "occupancy" |
| "formal_charges" | "formal_charge" |


The boolean syntax MolSysMT accepts includes the following words and symbols:


| Word | Symbol | Meaning |
|---|---|---|
| "and" | & | and |
| "or" | \| | or |
| "not" | ~ | not |
| "in" | | in |
|  | == | equal |
|  | != | not equal |
|  | < | less than |
|  | <= | less or equal than |
|  | > | greater than |
|  | >= | greater or equal than |


MolSysMT includes in its native selection syntax some other query restrictions involving spatial coordinates and bonds:


| Words | Meaning |
|---|---|
| "within ... of" | Elements are selected if they are at a certain distance of other elements |
| "not within ... of" | Elements are selected if they not are at a certain distance of other elements|
| "within ... with pbc of" | Elements are selected if they are at a certain distance of other elements with periodic boundary conditions|
| "within ... without pbc of" | Elements are selected if they are at a certain distance of other elements without periodic boundary conditions|


| Words | Meaning |
|---|---|
| "bonded to" | Elements are selected if they are bonded to other elements |
| "not bonded to" | Elements are selected if they are not bonded to other elements |


Finally, lists of elements can be queried with the help of the following expressions:


| Words | Meaning |
|---|---|
| "in groups of" | Elements are selected and returned in lists corresponding to different groups |
| "in components of" | Elements are selected and returned in lists corresponding to different components |
| "in molecules of" | Elements are selected and returned in lists corresponding to different molecules |
| "in chains of" | Elements are selected and returned in lists corresponding to different chains |
| "in entities of" | Elements are selected and returned in lists corresponding to different entities |


### Native selection shortcuts

In addition to the attribute names, MolSysMT’s native selection syntax includes a few convenient shortcuts:

| Word | Equivalent |
|---|---|
| "all" | Any *element* in the system, where *element* is given by the input argument `element` |
| "index" or "indices" | `*element*_index`, where *element* is given by the input argument `element` |
| "id" or "ids" | `*element*_id`, where *element* is given by the input argument `element` |
| "name" or "names" | `*element*_name`, where *element* is given by the input argument `element` |
| "type" or "types" | `*element*_type`, where *element* is given by the input argument `element` |

And some common atom-level shortcuts:

| Word | Equivalent |
|---|---|
| "backbone" | `atom_name in ['CA', 'N', 'C', 'O']` |
| "hydrogen" or "hydrogens" | `atom_type=='H'` |

### Your own selection shortcuts

You can customize the selection experience with your own selection shortcuts.

See {ref}`User_Intro_Customization`.

## Some examples of elements selections

Let's show here some examples of elements selections using the MolSysMT native syntax.

In [3]:
# Atoms of type C not named CA in groups "GLY"
view.whole.select('atom_type=="C" and not atom_name=="CA" and group_name=="GLY"')

[98, 191, 234, 246, 399, 437, 599, 833, 853, 869, 1232]

In [4]:
# Atoms belonging to groups named GLY or VAL in chain with name A.
select_atoms = view.whole.select('group_name in ["GLY","VAL"] and chain_name=="A"')
len(select_atoms)

107

In [5]:
# Bonds in group index 5
view.whole.select(element='bond', selection='group_index==5')

[45, 46, 47, 48, 50, 51, 52]

In [6]:
# A simple example: select all groups that belong to water molecules
water_groups = view.whole.select(element='group', selection='group_type=="water"')
len(water_groups)

136

In [7]:
# Select at a different level: molecules by chain id
view.whole.select(element='molecule', selection='chain_id=="C"')


[2]

You can include external Python variables in MolSysMT selection strings (Pandas query syntax) using the `@` prefix.

In [8]:
indices = list(range(10,30))
atoms = ["CA", "C", "O", "N"]
view.whole.select('atom_name==@atoms & atom_index==@indices')

[10, 11, 16, 17, 18, 19, 24, 25, 26, 27]

## Selecting elements "within a distance of"

MolSysMT can express spatial queries such as “elements of A within *d* of elements of B”.

The exact possibilities depend on your selection syntax, but the MolSysMT syntax supports a readable pattern like:

```
chain_name=="A" within 0.3 nm of chain_name=="B"
```

Here is how it looks in practice:

In [9]:
near_atoms = view.whole.select('atom_name=="O" within 0.6 nm of molecule_name=="BENZENE"')
len(near_atoms), near_atoms[:10]

(4, [653, 677, 765, 770])

## Selecting atoms "bonded to ..."

Atoms bonded to specific atoms can also be selected with `bonded to`:

In [10]:
atoms_selected = view.whole.select('(atom_type=="O" and chain_name=="A") bonded to (atom_type=="C" and chain_name=="A")')
len(atoms_selected)

240

## Selecting elements "in elements of ..."

This tool can also be used to find a list of elements lists fulfilling a query condition with the assistance of the strings: `groups of`, `components of`, `molecules of`, `chains of` or `entities of`. Let's see a couple of examples to understand how this option works.

In [11]:
view.whole.select('atom_type=="C" in groups of molecule_index==0 and group_name=="VAL"')

[[440, 441, 443, 444, 445],
 [555, 556, 558, 559, 560],
 [580, 581, 583, 584, 585],
 [675, 676, 678, 679, 680],
 [729, 730, 732, 733, 734],
 [797, 798, 800, 801, 802],
 [856, 857, 859, 860, 861],
 [1018, 1019, 1021, 1022, 1023],
 [1173, 1174, 1176, 1177, 1178]]

## Selections in viewer commands

Most scene operations accept a selection string. For example, you can hide water, isolate a ligand, or zoom to a region.

These commands are designed to be readable when you come back to the notebook weeks later.


In [12]:
# Focus on the benzene molecule
view.zoom('molecule_name=="BENZENE"')

## Selection inside regions

Regions are structural subsets of your system. When you call `region.select(...)`, the selection is interpreted as “within the region” (an intersection).

That is one of the reasons selections are so powerful: you can keep a region as a stable reference ("this binding site"), and then make more specific selections inside it.

Regions are documented in more depth in {doc}`../scene_management/regions`.
